# Full-Study Ablation Studies

Replicates the four pilot ablations (`ablation_results.json`) on the full dataset:

1. **Ablation 1 — Panel vs single judge**: per-model agreement (κ) between each individual
   judge and the majority-vote panel verdict. Tests whether panel-of-3 is worth it
   vs just using a single judge.
2. **Ablation 2 — Feature ablation**: trains a classifier with feature groups removed
   ("minus_X") or only one group ("only_X") to see which structural features matter.
3. **Ablation 3 — Training size sweep**: trains the classifier on 25%, 50%, 75%, 100%
   of available data to show learning curve / sample efficiency.
4. **Ablation 4 — Taxonomy granularity**: re-collapses the 6 H types into coarser
   schemas (3-axis: fabrication/omission/distortion; 2-axis: fab/non-fab; 1-axis: ANY)
   to show whether finer granularity helps or hurts detectability.

**Inputs:**
- `embeddings_openai.npy` — already cached from the detectability classifier notebook
- `embedding_index.csv` — preserves row alignment
- `panel_raw_judge_labels_full.csv` — per-judge labels (needed for ablation 1)
- `all_triplets_cache.csv` — model outputs (for hand-crafted features)

**Outputs (saved to `C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\`):**
- `ablation_results_full.json` — all four ablations
- `ablation_summary.txt` — paper-ready


In [1]:
import os, json, re
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import f1_score, roc_auc_score, cohen_kappa_score

OUTPUT_DIR    = r'C:\Opeyemi\PROMPTS\EVALUATION'
TRIPLETS_PATH = os.path.join(OUTPUT_DIR, 'all_triplets_cache.csv')
PANEL_RAW     = os.path.join(OUTPUT_DIR, 'panel_raw_judge_labels_full.csv')

HALL_DIR = r'C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS'
EMB_OPENAI_PATH = os.path.join(HALL_DIR, 'embeddings_openai.npy')
EMB_INDEX_PATH  = os.path.join(HALL_DIR, 'embedding_index.csv')

ABLATION_OUT = os.path.join(HALL_DIR, 'ablation_results_full.json')
SUMMARY_OUT  = os.path.join(HALL_DIR, 'ablation_summary.txt')

HCOLS = ['H1','H2','H3','H4','H5','H6']
RANDOM_SEED = 42

# Load embeddings (must run detectability classifier notebook first)
if not os.path.exists(EMB_OPENAI_PATH):
    raise SystemExit(f'Missing {EMB_OPENAI_PATH}. Run the detectability classifier notebook first.')

emb = np.load(EMB_OPENAI_PATH)
print(f'OpenAI embeddings: {emb.shape}')

# Load alignment index and merge with triplets + labels
idx = pd.read_csv(EMB_INDEX_PATH)
df_trip = pd.read_csv(TRIPLETS_PATH)
df_trip['model_output'] = df_trip['model_output'].fillna('').astype(str)
df_trip = df_trip.reset_index().rename(columns={'index': 'row_idx'})

labels = pd.read_csv(PANEL_RAW)
for h in HCOLS:
    labels[h] = pd.to_numeric(labels[h], errors='coerce')
clean_labels = labels[(labels[HCOLS] >= 0).all(axis=1)].copy()

def majority(col):
    return col.mode().iloc[0] if not col.mode().empty else 0

mv = clean_labels.groupby(['row_idx','model','technique','video','crime_type'])[HCOLS].agg(majority).reset_index()
mv['ANY'] = (mv[HCOLS].sum(axis=1) > 0).astype(int)

# Merge with triplets
data = df_trip.merge(mv[['row_idx'] + HCOLS + ['ANY']], on='row_idx', how='inner')
print(f'Merged rows (output + labels): {len(data):,}')

# Confirm alignment with embeddings: index file should match data row order
assert len(data) == len(emb), f'Mismatch: {len(data)} rows vs {len(emb)} embeddings'
print('Embeddings aligned with labeled rows.')


OpenAI embeddings: (9680, 1536)
Merged rows (output + labels): 9,680
Embeddings aligned with labeled rows.


---
## Ablation 1 — Panel vs single judge

For each individual judge (Claude / GPT / Gemini), measure agreement (κ) between that
judge's verdicts and the majority-vote panel verdict. If κ is high, a single judge could
substitute for the panel. If κ is much lower than 1, the panel adds value.

In [2]:
ablation1 = {}
for judge in ['Claude','GPT','Gemini']:
    judge_rows = clean_labels[clean_labels['judge'] == judge].copy()

    # Pivot: one row per row_idx with this judge's labels
    judge_lookup = judge_rows.set_index('row_idx')[HCOLS]

    per_h = {}
    for h in HCOLS:
        # Find row_idx values where both this judge AND the majority verdict exist
        common = judge_lookup.index.intersection(mv['row_idx'])
        if len(common) == 0:
            per_h[h] = {'note': 'no common rows'}
            continue
        # Align
        mv_idx = mv.set_index('row_idx').loc[common, h].values
        ju_idx = judge_lookup.loc[common, h].values
        # Cast to int
        mv_int = mv_idx.astype(int)
        ju_int = ju_idx.astype(int)
        agree = (mv_int == ju_int).mean() * 100
        try:
            k = cohen_kappa_score(mv_int, ju_int)
        except Exception:
            k = None
        per_h[h] = {
            'n': int(len(common)),
            'agree_pct': float(agree),
            'kappa': float(k) if k is not None else None,
        }

    # Also ANY
    any_lookup = (judge_lookup[HCOLS].sum(axis=1) > 0).astype(int)
    common = any_lookup.index.intersection(mv['row_idx'])
    if len(common):
        mv_any = mv.set_index('row_idx').loc[common, 'ANY'].values.astype(int)
        ju_any = any_lookup.loc[common].values
        agree = (mv_any == ju_any).mean() * 100
        try:
            k = cohen_kappa_score(mv_any, ju_any)
        except Exception:
            k = None
        per_h['ANY'] = {
            'n': int(len(common)),
            'agree_pct': float(agree),
            'kappa': float(k) if k is not None else None,
        }

    ablation1[judge] = per_h
    print(f'\n{judge} vs Panel (majority):')
    for h, v in per_h.items():
        if 'kappa' in v and v['kappa'] is not None:
            print(f'  {h}: n={v["n"]:>5,}  agree={v["agree_pct"]:>5.1f}%  kappa={v["kappa"]:.3f}')
        else:
            print(f'  {h}: {v}')

# Summary: would a single judge suffice?
print('\n=== Macro kappa (single judge vs panel majority) ===')
for judge in ['Claude','GPT','Gemini']:
    ks = [ablation1[judge][h]['kappa'] for h in HCOLS
          if 'kappa' in ablation1[judge][h] and ablation1[judge][h]['kappa'] is not None]
    if ks:
        print(f'  {judge}: macro kappa = {np.mean(ks):.3f}')



Claude vs Panel (majority):
  H1: n=6,455  agree= 97.6%  kappa=0.950
  H2: n=6,455  agree= 93.0%  kappa=0.749
  H3: n=6,455  agree= 93.6%  kappa=0.872
  H4: n=6,455  agree= 82.4%  kappa=0.647
  H5: n=6,455  agree= 98.2%  kappa=0.964
  H6: n=6,455  agree= 92.7%  kappa=0.750
  ANY: n=6,455  agree= 95.4%  kappa=0.848

GPT vs Panel (majority):
  H1: n=6,452  agree= 75.8%  kappa=0.284
  H2: n=6,452  agree= 90.6%  kappa=0.767
  H3: n=6,452  agree= 93.6%  kappa=0.862
  H4: n=6,452  agree= 94.0%  kappa=0.834
  H5: n=6,452  agree= 81.2%  kappa=0.298
  H6: n=6,452  agree= 87.6%  kappa=0.744
  ANY: n=6,452  agree= 90.3%  kappa=0.132

Gemini vs Panel (majority):
  H1: n=6,451  agree= 80.4%  kappa=0.569
  H2: n=6,451  agree= 98.7%  kappa=0.960
  H3: n=6,451  agree= 95.1%  kappa=0.901
  H4: n=6,451  agree= 95.7%  kappa=0.886
  H5: n=6,451  agree= 83.7%  kappa=0.642
  H6: n=6,451  agree= 97.3%  kappa=0.930
  ANY: n=6,451  agree= 91.3%  kappa=0.363

=== Macro kappa (single judge vs panel majority) ==

---
## Ablation 2 — Hand-crafted feature ablation

Replicates pilot's `ablation2_features`. Tests how much detectability comes from each
feature group (verbosity, hedging, model-id, multi-turn).

In [3]:
# Re-compute hand-crafted features (same definitions as detectability notebook)
HEDGE_WORDS = [
    ' may ',' might ',' possibly ',' perhaps ',' likely ',' probably ',
    ' appears ',' seems ',' suggests ',' suggesting ',' could ',
    ' presumably ',' apparently ',' uncertain',' approximately ',
    ' roughly ',' somewhat ',' arguably ',
]
SENT_RE  = re.compile(r'[.!?]+(?:\s|$)')
CLAIM_RE = re.compile(r'(?:[.!?;]|--|—|\n\s*[-*•])')

def hand_features(text, model_name, technique):
    if not isinstance(text, str): text = ''
    t = text.lower()
    nw = len(text.split())
    n_sent  = sum(1 for p in SENT_RE.split(text) if p.strip())
    n_claim = sum(1 for p in CLAIM_RE.split(text) if len(p.split()) >= 3)
    n_hedge = sum(t.count(h) for h in HEDGE_WORDS)
    return {
        'verbosity_words': nw,
        'verbosity_log':   np.log1p(nw),
        'hedging_density': n_hedge / max(nw, 1),
        'claim_density':   n_claim / max(nw, 1),
        'sent_density':    n_sent / max(nw, 1),
        'is_claude':       int(model_name == 'Claude'),
        'is_gpt':          int(model_name == 'GPT'),
        'is_gemini':       int(model_name == 'Gemini'),
        'is_zero_shot':    int(technique == 'Zero-Shot'),
        'is_sequential':   int(technique == 'Sequential'),
        'is_least_to_most':int(technique == 'Least-to-Most'),
        'is_react':        int(technique == 'ReAct'),
        'is_multiturn':    int(technique in ('Sequential','Least-to-Most','ReAct')),
    }

print('Computing hand features...')
feats = [hand_features(r['model_output'], r['model'], r['technique']) for _, r in data.iterrows()]
hand_df = pd.DataFrame(feats)
print(f'Hand features: {hand_df.shape}')

# Define feature ablation groups
verbosity_cols  = [c for c in hand_df.columns if 'verbosity' in c or c == 'sent_density'
                   or c == 'claim_density']
hedging_cols    = [c for c in hand_df.columns if 'hedging' in c]
model_id_cols   = ['is_claude','is_gpt','is_gemini']
multiturn_cols  = ['is_zero_shot','is_sequential','is_least_to_most','is_react','is_multiturn']

CONFIGS = {
    'full':              list(hand_df.columns),
    'minus_verbosity':   [c for c in hand_df.columns if c not in verbosity_cols],
    'minus_hedging':     [c for c in hand_df.columns if c not in hedging_cols],
    'minus_model_id':    [c for c in hand_df.columns if c not in model_id_cols],
    'minus_multiturn':   [c for c in hand_df.columns if c not in multiturn_cols],
    'only_verbosity':    verbosity_cols,
    'only_hedging':      hedging_cols,
    'only_model_id':     model_id_cols,
    'only_multiturn':    multiturn_cols,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

def cv_auc_f1(X, y):
    if y.sum() < 10 or y.sum() > len(y) - 10:
        return None, None
    Xs = StandardScaler().fit_transform(X)
    try:
        scores = cross_validate(
            LogisticRegression(max_iter=2000, class_weight='balanced',
                               random_state=RANDOM_SEED),
            Xs, y, cv=skf,
            scoring=('f1','roc_auc'), n_jobs=-1, error_score='raise',
        )
        return float(np.mean(scores['test_f1'])), float(np.mean(scores['test_roc_auc']))
    except Exception:
        return None, None

ablation2 = {}
for name, cols in CONFIGS.items():
    if not cols:
        continue
    X = hand_df[cols].values.astype(np.float32)
    out = {'config': cols}
    print(f'\n  {name} ({len(cols)} cols):')
    for h in HCOLS + ['ANY']:
        y = data[h].astype(int).values
        f1, auc = cv_auc_f1(X, y)
        out[h] = {'f1': f1, 'auc': auc, 'n_pos': int(y.sum())}
        if auc is not None:
            print(f'    {h}: AUC={auc:.3f}  F1={f1:.3f}')
        else:
            print(f'    {h}: imbalanced')
    ablation2[name] = out


Computing hand features...
Hand features: (9680, 13)

  full (13 cols):
    H1: AUC=0.815  F1=0.721
    H2: AUC=0.688  F1=0.380
    H3: AUC=0.701  F1=0.587
    H4: AUC=0.720  F1=0.506
    H5: AUC=0.824  F1=0.779
    H6: AUC=0.765  F1=0.526
    ANY: AUC=0.741  F1=0.685

  minus_verbosity (9 cols):
    H1: AUC=0.801  F1=0.690
    H2: AUC=0.675  F1=0.370
    H3: AUC=0.696  F1=0.584
    H4: AUC=0.701  F1=0.488
    H5: AUC=0.800  F1=0.761
    H6: AUC=0.745  F1=0.513
    ANY: AUC=0.727  F1=0.634

  minus_hedging (12 cols):
    H1: AUC=0.812  F1=0.720
    H2: AUC=0.687  F1=0.380
    H3: AUC=0.702  F1=0.587
    H4: AUC=0.717  F1=0.489
    H5: AUC=0.823  F1=0.774
    H6: AUC=0.766  F1=0.525
    ANY: AUC=0.741  F1=0.680

  minus_model_id (10 cols):
    H1: AUC=0.754  F1=0.713
    H2: AUC=0.636  F1=0.339
    H3: AUC=0.698  F1=0.593
    H4: AUC=0.715  F1=0.491
    H5: AUC=0.753  F1=0.762
    H6: AUC=0.725  F1=0.494
    ANY: AUC=0.700  F1=0.743

  minus_multiturn (8 cols):
    H1: AUC=0.810  F1=0.7

---
## Ablation 3 — Training size sweep

Train on 25%, 50%, 75%, 100% of training data and evaluate on a fixed held-out test set.
Shows whether the classifier is data-limited or already saturated.

In [4]:
# Stratified 80/20 split for the held-out evaluation; vary the training fraction within
# the 80% training portion. Use OpenAI embeddings as the feature set (best from the
# detectability run).

from sklearn.metrics import roc_auc_score, f1_score

# Pre-split: stratified on ANY so all training fractions test on the same held-out set
y_any = data['ANY'].astype(int).values
X_train_full, X_test, idx_train, idx_test = train_test_split(
    emb, np.arange(len(data)), test_size=0.2, random_state=RANDOM_SEED, stratify=y_any,
)

ablation3 = {}
for frac in (0.25, 0.50, 0.75, 1.00):
    # Subsample within training set
    n_take = int(len(X_train_full) * frac)
    rng = np.random.default_rng(RANDOM_SEED)
    sub = rng.choice(len(X_train_full), size=n_take, replace=False)
    X_tr_sub = X_train_full[sub]
    idx_tr_sub = idx_train[sub]

    out = {'fraction': frac, 'n_train': int(n_take)}
    print(f'\n  fraction={frac}  n_train={n_take:,}')
    for h in HCOLS + ['ANY']:
        y_full = data[h].astype(int).values
        y_tr = y_full[idx_tr_sub]
        y_te = y_full[idx_test]
        if y_tr.sum() < 10 or y_te.sum() < 5:
            out[h] = {'note': f'imbalanced (train_pos={int(y_tr.sum())}, test_pos={int(y_te.sum())})'}
            continue
        scaler = StandardScaler().fit(X_tr_sub)
        Xtr_s = scaler.transform(X_tr_sub)
        Xte_s = scaler.transform(X_test)
        clf = LogisticRegression(max_iter=2000, class_weight='balanced',
                                  random_state=RANDOM_SEED)
        clf.fit(Xtr_s, y_tr)
        ypred = clf.predict(Xte_s)
        try:
            yprob = clf.predict_proba(Xte_s)[:, 1]
            auc = float(roc_auc_score(y_te, yprob))
        except Exception:
            auc = None
        f1 = float(f1_score(y_te, ypred, zero_division=0))
        out[h] = {'f1': f1, 'auc': auc, 'n_pos': int(y_tr.sum())}
        if auc is not None:
            print(f'    {h}: AUC={auc:.3f}  F1={f1:.3f}  n_pos_train={int(y_tr.sum())}')

    ablation3[f'frac_{frac:.2f}'] = out



  fraction=0.25  n_train=1,936
    H1: AUC=0.804  F1=0.753  n_pos_train=1045
    H2: AUC=0.757  F1=0.429  n_pos_train=351
    H3: AUC=0.802  F1=0.680  n_pos_train=785
    H4: AUC=0.801  F1=0.569  n_pos_train=492
    H5: AUC=0.775  F1=0.761  n_pos_train=1147
    H6: AUC=0.772  F1=0.526  n_pos_train=476
    ANY: AUC=0.745  F1=0.880  n_pos_train=1653

  fraction=0.5  n_train=3,872
    H1: AUC=0.811  F1=0.749  n_pos_train=2080
    H2: AUC=0.776  F1=0.461  n_pos_train=707
    H3: AUC=0.802  F1=0.669  n_pos_train=1555
    H4: AUC=0.801  F1=0.575  n_pos_train=949
    H5: AUC=0.787  F1=0.772  n_pos_train=2317
    H6: AUC=0.780  F1=0.508  n_pos_train=906
    ANY: AUC=0.727  F1=0.864  n_pos_train=3309

  fraction=0.75  n_train=5,808
    H1: AUC=0.827  F1=0.769  n_pos_train=3169
    H2: AUC=0.796  F1=0.486  n_pos_train=1065
    H3: AUC=0.821  F1=0.695  n_pos_train=2300
    H4: AUC=0.807  F1=0.580  n_pos_train=1434
    H5: AUC=0.811  F1=0.781  n_pos_train=3490
    H6: AUC=0.789  F1=0.535  n_pos_t

---
## Ablation 4 — Taxonomy granularity

Compares classifier performance under different label collapsing schemes:

- 6-type: original H1..H6
- 3-type: fabrication (H1+H5+H6) / omission (H3) / distortion (H2+H4)
- 2-type: fabrication (H1+H5+H6) / non-fabrication (H2+H3+H4)
- Binary: ANY (any H)

In [5]:
GRANULARITIES = {
    '6-type (full H1-H6)': {
        'H1': ['H1'], 'H2': ['H2'], 'H3': ['H3'],
        'H4': ['H4'], 'H5': ['H5'], 'H6': ['H6'],
    },
    '3-type (fabrication / omission / distortion)': {
        'FABRICATION': ['H1','H5','H6'],
        'OMISSION':    ['H3'],
        'DISTORTION':  ['H2','H4'],
    },
    '2-type (fabrication / non-fabrication)': {
        'FABRICATION':     ['H1','H5','H6'],
        'NON_FABRICATION': ['H2','H3','H4'],
    },
    'Binary (any hallucination)': {
        'ANY': ['H1','H2','H3','H4','H5','H6'],
    },
}

# Re-use the same train/test split from ablation 3 for consistency
ablation4 = {}
for gran_name, targets in GRANULARITIES.items():
    print(f'\n  {gran_name}:')
    out = {}
    f1s, aucs = [], []
    for tname, components in targets.items():
        # OR across components
        y_full = (data[components].sum(axis=1) > 0).astype(int).values
        y_tr = y_full[idx_train]
        y_te = y_full[idx_test]
        if y_tr.sum() < 10 or y_te.sum() < 5 or y_tr.sum() > len(y_tr) - 10:
            out[tname] = {'components': components, 'note': 'imbalanced'}
            continue
        scaler = StandardScaler().fit(X_train_full)
        Xtr_s = scaler.transform(X_train_full)
        Xte_s = scaler.transform(X_test)
        clf = LogisticRegression(max_iter=2000, class_weight='balanced',
                                  random_state=RANDOM_SEED)
        clf.fit(Xtr_s, y_tr)
        ypred = clf.predict(Xte_s)
        try:
            yprob = clf.predict_proba(Xte_s)[:, 1]
            auc = float(roc_auc_score(y_te, yprob))
        except Exception:
            auc = None
        f1 = float(f1_score(y_te, ypred, zero_division=0))
        out[tname] = {
            'components': components,
            'n_pos': int(y_full.sum()),
            'n_neg': int(len(y_full) - y_full.sum()),
            'f1': f1, 'auc': auc,
        }
        if auc is not None:
            aucs.append(auc); f1s.append(f1)
            print(f'    {tname}: AUC={auc:.3f}  F1={f1:.3f}  '
                  f'n_pos={int(y_full.sum())}/{len(y_full)}')
    out['_summary'] = {
        'n_targets': len(targets),
        'macro_f1':  float(np.mean(f1s)) if f1s else None,
        'macro_auc': float(np.mean(aucs)) if aucs else None,
    }
    if out['_summary']['macro_auc']:
        print(f'    --> macro AUC = {out["_summary"]["macro_auc"]:.3f}')
    ablation4[gran_name] = out



  6-type (full H1-H6):
    H1: AUC=0.844  F1=0.776  n_pos=5282/9680
    H2: AUC=0.802  F1=0.498  n_pos=1787/9680
    H3: AUC=0.841  F1=0.713  n_pos=3848/9680
    H4: AUC=0.823  F1=0.620  n_pos=2426/9680
    H5: AUC=0.824  F1=0.782  n_pos=5839/9680
    H6: AUC=0.795  F1=0.534  n_pos=2294/9680
    --> macro AUC = 0.822

  3-type (fabrication / omission / distortion):
    FABRICATION: AUC=0.851  F1=0.823  n_pos=6351/9680
    OMISSION: AUC=0.841  F1=0.713  n_pos=3848/9680
    DISTORTION: AUC=0.774  F1=0.650  n_pos=3783/9680
    --> macro AUC = 0.822

  2-type (fabrication / non-fabrication):
    FABRICATION: AUC=0.851  F1=0.823  n_pos=6351/9680
    NON_FABRICATION: AUC=0.781  F1=0.728  n_pos=5015/9680
    --> macro AUC = 0.816

  Binary (any hallucination):
    ANY: AUC=0.760  F1=0.855  n_pos=8305/9680
    --> macro AUC = 0.760


---
## Save results + paper-ready summary

In [6]:
ablation_results = {
    'ablation1_panel_vs_single':    ablation1,
    'ablation2_features':            ablation2,
    'ablation3_training_size':       ablation3,
    'ablation4_taxonomy_granularity': ablation4,
}
with open(ABLATION_OUT, 'w') as f:
    json.dump(ablation_results, f, indent=2)
print(f'Saved: {ABLATION_OUT}')

# Build summary
lines = []
lines.append('=' * 76)
lines.append('FULL-STUDY ABLATION RESULTS — SUMMARY')
lines.append('=' * 76)

# Ablation 1 summary
lines.append('')
lines.append('-- ABLATION 1: Single judge vs panel-of-3 (macro kappa) --')
for judge in ['Claude','GPT','Gemini']:
    ks = [ablation1[judge][h]['kappa'] for h in HCOLS
          if ablation1[judge][h].get('kappa') is not None]
    if ks:
        lines.append(f'  {judge:<8}  macro kappa = {np.mean(ks):.3f}  (per-H range: '
                     f'{min(ks):.3f}-{max(ks):.3f})')
lines.append('  Interpretation: high kappa (>0.85) means single judge could substitute for panel.')

# Ablation 2 summary
lines.append('')
lines.append('-- ABLATION 2: Feature ablation (5-fold CV AUC for ANY) --')
for cfg, res in ablation2.items():
    auc_any = res.get('ANY', {}).get('auc')
    if auc_any is not None:
        lines.append(f'  {cfg:<22}  AUC(ANY) = {auc_any:.3f}')

# Ablation 3
lines.append('')
lines.append('-- ABLATION 3: Training size sweep (held-out AUC for ANY) --')
for k in ['frac_0.25','frac_0.50','frac_0.75','frac_1.00']:
    if k in ablation3:
        v = ablation3[k]
        any_auc = v.get('ANY', {}).get('auc')
        n = v.get('n_train', 0)
        if any_auc is not None:
            lines.append(f'  {k}: n={n:>5,}  AUC(ANY) = {any_auc:.3f}')

# Ablation 4
lines.append('')
lines.append('-- ABLATION 4: Taxonomy granularity (held-out macro AUC) --')
for gran, res in ablation4.items():
    summary = res.get('_summary', {})
    macro_auc = summary.get('macro_auc')
    macro_f1  = summary.get('macro_f1')
    if macro_auc is not None:
        lines.append(f'  {gran:<48} macro AUC = {macro_auc:.3f}  macro F1 = {macro_f1:.3f}')

summary = '\n'.join(lines)
with open(SUMMARY_OUT, 'w') as f:
    f.write(summary)
print('\n' + summary)
print(f'\nSaved: {SUMMARY_OUT}')


Saved: C:\Opeyemi\PROMPTS\HALLUCINATION-METRICS\ablation_results_full.json

FULL-STUDY ABLATION RESULTS — SUMMARY

-- ABLATION 1: Single judge vs panel-of-3 (macro kappa) --
  Claude    macro kappa = 0.822  (per-H range: 0.647-0.964)
  GPT       macro kappa = 0.631  (per-H range: 0.284-0.862)
  Gemini    macro kappa = 0.815  (per-H range: 0.569-0.960)
  Interpretation: high kappa (>0.85) means single judge could substitute for panel.

-- ABLATION 2: Feature ablation (5-fold CV AUC for ANY) --
  full                    AUC(ANY) = 0.741
  minus_verbosity         AUC(ANY) = 0.727
  minus_hedging           AUC(ANY) = 0.741
  minus_model_id          AUC(ANY) = 0.700
  minus_multiturn         AUC(ANY) = 0.735
  only_verbosity          AUC(ANY) = 0.699
  only_hedging            AUC(ANY) = 0.561
  only_model_id           AUC(ANY) = 0.684
  only_multiturn          AUC(ANY) = 0.580

-- ABLATION 3: Training size sweep (held-out AUC for ANY) --
  frac_0.25: n=1,936  AUC(ANY) = 0.745
  frac_0.50: n